# Core MCP Feature

## 1. Sampling

Sampling allows **a server to access a language model like Claude through a connected MCP client**. Instead of the server directly calling Claude, it asks the client to make the call on its behalf. This shifts the responsibility and cost of text generation from the server to the client.

![](./figure/09/Sampling_03.png)
![](./figure/09/Sampling_04.png)

### The Problem Sampling Solves

Imagine you have an MCP server with a research tool that fetches information from Wikipedia. After gathering all that data, you need to summarize it into a coherent report. You have two options:

1. Give the MCP server direct access to Claude. The server would need *its own API key, handle authentication, manage costs, and implement all the Claude integration code*. This works but **adds significant complexity**.

2. Use sampling. The server generates a prompt and asks the client "Could you call Claude for me?" The client, which already has a connection to Claude, makes the call and returns the results.

![](./figure/09/Sampling_01.png)
![](./figure/09/Sampling_02.png)

### How Sampling Works

The flow is straightforward:

1. Server completes its work (like fetching Wikipedia articles)
2. Server creates a prompt asking for text generation
3. Server sends a sampling request to the client
4. Client calls Claude with the provided prompt
5. Client returns the generated text to the server
6. Server uses the generated text in its response

### Benefits of Sampling

- **Reduces server complexity** : The server doesn't need to integrate with language models directly
- **Shifts cost burden** : The client pays for token usage, not the server
- **No API keys needed** : The server doesn't need credentials for Claude
- **Perfect for public servers** : You don't want a public server racking up AI costs for every user

### Implementation

Setting up sampling requires code on both sides:

#### 1. Server Side
In your tool function, use the `create_message` function to request text generation:

```python
@mcp.tool()
async def summarize(text_to_summarize: str, ctx: Context):
    prompt = f"""
    Please summarize the following text:
    {text_to_summarize}
    """
    
    result = await ctx.session.create_message(
        messages=[
            SamplingMessage(
                role="user",
                content=TextContent(
                    type="text",
                    text=prompt
                )
            )
        ],
        max_tokens=4000,
        system_prompt="You are a helpful research assistant",
    )
    
    if result.content.type == "text":
        return result.content.text
    else:
        raise ValueError("Sampling failed")
```

#### 2. Client Side
Create a sampling callback that handles the server's requests:

```python
async def sampling_callback(
    context: RequestContext, params: CreateMessageRequestParams
):
    # Call Claude using the Anthropic SDK
    text = await chat(params.messages)
    
    return CreateMessageResult(
        role="assistant",
        model=model,
        content=TextContent(type="text", text=text),
    )
```

Then pass this callback when initializing your client session:

```python
async with ClientSession(
    read,
    write,
    sampling_callback=sampling_callback
) as session:
    await session.initialize()
```

### When to Use Sampling

Sampling is most valuable when building publicly accessible MCP servers. You don't want random users generating unlimited text at your expense. By using sampling, each client pays for their own AI usage while still benefiting from your server's functionality.

The technique essentially moves the AI integration complexity from your server to the client, which often already has the necessary connections and credentials in place.

## 2. Sampling Walkthrough
code 在 [./sampling](./sampling/)

### Initiating sampling

On the server, during a tool call, run the `create_message()` method, passing in some messages that you wish to send to a language model.

```python
# server.py
@mcp.tool()
async def summarize(text_to_summarize: str, ctx: Context):
    prompt = f"""
        Please summarize the following text:
        {text_to_summarize}
    """

    ''' ============================================================================ '''
    result = await ctx.session.create_message(
        messages=[
            SamplingMessage(
                role="user", content=TextContent(type="text", text=prompt)
            )
        ],
        max_tokens=4000,
        system_prompt="You are a helpful research assistant.",
    )
    ''' ============================================================================ '''

    if result.content.type == "text":
        return result.content.text
    else:
        raise ValueError("Sampling failed")
```

### Sampling callback

On the client, you must implement a sampling callback. It will receive a list of messages provided by the server.

```python
# client.py
''' ============================================================================ '''
async def sampling_callback(
    context: RequestContext, params: CreateMessageRequestParams
):
''' ============================================================================ '''
    # Call Claude using the Anthropic SDK
    text = await chat(params.messages)

    return CreateMessageResult(
        role="assistant",
        model=model,
        content=TextContent(type="text", text=text),
    )
```

### Message formats

The list of messages provided by the server are formatted for communication in MCP. The individual messages aren't guaranteed to be compatible with whatever LLM SDK you are using.

For example, if you're using the Anthropic SDK, you'll have to write a little bit of conversion logic to turn the MCP messages into a format compatible with Anthropic's SDK.
> 看使用哪個 LLM，需要將其調整為該 model 能接受的 format

```python
# client.py
async def chat(input_messages: list[SamplingMessage], max_tokens=4000):
    messages = []
    ''' ============================================================================ '''
    for msg in input_messages:
        if msg.role == "user" and msg.content.type == "text":
            content = (
                msg.content.text
                if hasattr(msg.content, "text")
                else str(msg.content)
            )
            messages.append({"role": "user", "content": content})
        elif msg.role == "assistant" and msg.content.type == "text":
            content = (
                msg.content.text
                if hasattr(msg.content, "text")
                else str(msg.content)
            )
            messages.append({"role": "assistant", "content": content})
    ''' ============================================================================ '''
    response = await anthropic_client.messages.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
    )

    text = "".join([p.text for p in response.content if p.type == "text"])
    return text
```

### Returning generated text

After generating text with the LLM, you'll return a `CreateMessageResult`, which contains the generated text.

```python
# client.py
async def sampling_callback(
    context: RequestContext, params: CreateMessageRequestParams
):
    ''' ============================================================================ '''
    # Call Claude using the Anthropic SDK
    text = await chat(params.messages)

    return CreateMessageResult(
        role="assistant",
        model=model,
        content=TextContent(type="text", text=text),
    )
    ''' ============================================================================ '''
```

### Connecting the callback

Don't forget: the callback on the client needs to be passed into the `ClientSession` call.

```python
# client.py
async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            ''' ============================================================================ '''
            read, write, sampling_callback=sampling_callback
            ''' ============================================================================ '''
        ) as session:
            await session.initialize()

            result = await session.call_tool(
                name="summarize",
                arguments={"text_to_summarize": "lots of text"},
            )
            print(result.content)
```

### Getting the result

After the client has generated and returned some text, it will be sent to the server. You can do anything with this text:

- Use it as part of a workflow in your tool
- Decide to make another sampling call
- Return the generated text

```python
# server.py
@mcp.tool()
async def summarize(text_to_summarize: str, ctx: Context):
    prompt = f"""
        Please summarize the following text:
        {text_to_summarize}
    """

    result = await ctx.session.create_message(
        messages=[
            SamplingMessage(
                role="user", content=TextContent(type="text", text=prompt)
            )
        ],
        max_tokens=4000,
        system_prompt="You are a helpful research assistant.",
    )
    ''' ============================================================================ '''
    if result.content.type == "text":
        return result.content.text
    else:
        raise ValueError("Sampling failed")
    ''' ============================================================================ '''
```

### 路線圖

In [ ]:
[Server]                              [Client]
  ↓
`summarize()` 被呼叫
  ↓
`ctx.session.create_message()`  ──────→  MCP 協議路由 (知道要轉交給 `sampling_callback()` 處理，因為在 `ClientSession(..., sampling_callback=sampling_callback)` 有註冊)
(MCP 框架內部發出                            |
`sampling/createMessage` request)           |
  ↓ (await，暫停等待)                        ↓
                                      `sampling_callback()` 被自動呼叫
                                            ↓
                                      `chat()` 轉換格式 + 呼叫 Anthropic API
                                            ↓
                                      return `CreateMessageResult(...)`
  ↓ (await 結束，拿到 result)  ←──────    MCP 協議路由回傳
`result.content.text`
  ↓
return 給最初呼叫者

## 3. Log and progress notifications

Logging and progress notifications are simple to implement but make a huge difference in **User Experience** when working with MCP servers. They help users understand what's happening during long-running operations instead of wondering if something has broken.

When Claude calls a tool that takes time to complete - like researching a topic or processing data - users typically see nothing until the operation finishes. This can be frustrating because they don't know if the tool is working or has stalled.

With logging and progress notifications enabled, users get real-time feedback showing exactly what's happening behind the scenes. They can see progress bars, status messages, and detailed logs as the operation runs.

![](./figure/09/log_and_progresss_01.png)
![](./figure/09/log_and_progresss_02.png)

### How It Works

In the Python MCP SDK, logging and progress notifications work through the `Context` argument that's **automatically provided to your tool functions**. This context object gives you methods to communicate back to the client during execution.

The key methods you'll use are:

1. `context.info()` : Send log messages to the client
2. `context.report_progress()` : Update progress with current and total values

```python
@mcp.tool(
    name="research",
    description="Research a given topic"
)
async def research(
    topic: str = Field(description="Topic to research"),
    *,
    context: Context
):
    await context.info("About to do research...") # <--
    await context.report_progress(20, 100) # <--
    sources = await do_research(topic)
    
    await context.info("Writing report...") # <--
    await context.report_progress(70, 100) # <--
    results = await generate_report(sources)
    
    return results
```

### Client-Side Implementation

On the client side, you need to set up *callback* functions to handle these notifications. The server emits these messages, but it's up to your client application to decide **how to present** them to users.

You provide the **logging callback** when *creating the client session*, and the **progress callback** when *making individual tool calls*. This gives you flexibility to handle different types of notifications appropriately.

```python
async def logging_callback(params: LoggingMessageNotificationParams):
    print(params.data)

async def print_progress_callback(
    progress: float, total: float | None, message: str | None
):
    if total is not None:
        percentage = (progress / total) * 100
        print(f"Progress: {progress}/{total} ({percentage:.1f}%)")
    else:
        print(f"Progress: {progress}")

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            read,
            write,
            logging_callback=logging_callback # <-- Setup log message
        ) as session:
            await session.initialize()
            
            await session.call_tool(
                name="add",
                arguments={"a": 1, "b": 3},
                progress_callback=print_progress_callback, # <-- Setup progress message
            )
```

### Presentation Options

How you present these notifications depends on your application type:

- **CLI applications** : Simply print messages and progress to the terminal
- **Web applications** : Use WebSockets, server-sent events, or polling to push updates to the browser
- **Desktop applications** : Update progress bars and status displays in your UI

Remember that implementing these notifications is entirely optional. You can choose to ignore them completely, show only certain types, or present them however makes sense for your application. They're purely user experience enhancements to help users understand what's happening during long-running operations.

## 4. Notifications walkthrough
code 在 [./notifications](./notifications)

### Tool function receives Context argument

Tool functions automatically receive `Context` as their last argument. This object has methods for logging and reporting progress to the client.

```python
# server.py
@mcp.tool()
'''================================================='''
async def add(a: int, b: int, ctx: Context) -> int:
'''================================================='''
    await ctx.info("Preparing to add...")
    await ctx.report_progress(20, 100)

    await asyncio.sleep(2)

    await ctx.info("OK, adding...")
    await ctx.report_progress(80, 100)

    return a + b
```

### Create logs and progess with context

Throughout your tool function, call the `info()`, `warning()`, `debug()`, or `error()` methods to log **different types of messages** for the client. Also call the `report_progress()` method to estimate the amount of remaining work for the tool call.

```python
# server.py
@mcp.tool()
async def add(a: int, b: int, ctx: Context) -> int:
'''================================================='''
    await ctx.info("Preparing to add...")
    await ctx.report_progress(20, 100)

    await asyncio.sleep(2)

    await ctx.info("OK, adding...")
    await ctx.report_progress(80, 100)
'''================================================='''
    return a + b
```

### Define callbacks on the client

The client needs to define logging and progress callbacks, which will automatically be called whenever the server emits log or progress messages. These callbacks should try to display the provided logging and progress data to the user.

```python
# client.py
'''=================================================================='''
async def logging_callback(params: LoggingMessageNotificationParams):
    print(params.data)

async def print_progress_callback(
    progress: float, total: float | None, message: str | None
):
    if total is not None:
        percentage = (progress / total) * 100
        print(f"Progress: {progress}/{total} ({percentage:.1f}%)")
    else:
        print(f"Progress: {progress}")
'''=================================================================='''
```

### Pass callbacks to appropriate functions

Make sure you provide the **logging callback** to the `ClientSession` and the **progress callback** to the `call_tool()`function.

```python
# client.py
'''=================================================================='''
async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            read, write, logging_callback=logging_callback
        ) as session:
            await session.initialize()

            await session.call_tool(
                name="add",
                arguments={"a": 1, "b": 3},
                progress_callback=print_progress_callback,
            )
'''=================================================================='''
```

## 5. Roots

Roots are a way to **grant MCP servers access to specific files and folders on your local machine**. Think of them as a permission system that says "Hey, MCP server, you can access these files" - but they do much more than just grant permission.

![](./figure/09/Roots_03.png)

### The Problem Roots Solve

Without roots, you'd run into a common issue. Imagine you have an MCP server with a video conversion tool that takes a file path and converts an MP4 to MOV format.

![](./figure/09/Roots_01.png)

When a user asks Claude to "convert biking.mp4 to mov format", Claude would call the tool with just the filename. But here's the problem : **Claude has no way to search through your entire file system to find where that file actually lives**.

![](./figure/09/Roots_02.png)

Your file system might be complex with files scattered across different directories. The user knows the `biking.mp4` file is in their Movies folder, but Claude doesn't have that context.

You could solve this by requiring users to always provide full paths, but that's not very user-friendly. Nobody wants to type out complete file paths every time.

### Roots in Action

Here's how the workflow changes with roots:

1. User asks to convert a video file
2. Claude calls `list_roots` to see what directories it can access
3. Claude calls `read_dir` on accessible directories to find the file
4. Once found, Claude calls the conversion tool with the full path

This happens automatically - users can still just say "convert biking.mp4" without providing full paths.

### Security and Boundaries

Roots also **provide security by limiting access**. If you only grant access to your `Desktop` folder, the MCP server cannot access files in other locations like `Documents` or `Downloads`.

When Claude tries to access a file outside the approved roots, it gets an error and can inform the user that the file isn't accessible from the current server configuration.
> 用 LLM 進行 limit 確定能完全限制嗎? 還是有方法可以 bypass?

### Implementation Details

The MCP SDK **doesn't automatically enforce root restrictions** - you need to implement this yourself. A typical pattern is to create a helper function like `is_path_allowed()` that:

- Takes a requested file path
- Gets the list of approved roots
- Checks if the requested path falls within one of those roots
- Returns true/false for access permission

You then call this function in any tool that accesses files or directories before performing the actual file operation.
> 要自己寫的話有沒有 recom example 可以參考，自己寫感覺很危險

### Key Benefits

- **User-friendly** : Users don't need to provide full file paths
- **Focused search** : Claude only looks in approved directories, making file discovery faster
- **Security** : Prevents accidental access to sensitive files outside approved areas
- **Flexibility** : You can provide roots through tools or inject them directly into prompts

Roots make MCP servers both more powerful and more secure by giving Claude the context it needs to find files while maintaining clear boundaries around what it can access.

## 6. Roots walkthrough
code 在 [./roots](./roots)

### Defining roots

Ideally, a user will dictate which files/folders can be accessed by the MCP server.

This program is set up to accept a list of CLI arguments, which are interpretted as paths that the user wants to allow access to.

That list of paths is provided to the `MCPClient` down on lines `42`.

```python
# main.py
async def main():
    claude_service = Claude(model=claude_model)
    '''================================================'''
    # Get root directories from command line arguments
    root_paths = sys.argv[1:]
    '''================================================'''
    if not root_paths:
        print("Usage: uv run main.py <root1> [root2] ...")
        print("Example: uv run main.py /path/to/videos /another/path")
        sys.exit(1)

    clients = {}

    async with AsyncExitStack() as stack:
        # Create the MCP client with the provided root directories
        doc_client = await stack.enter_async_context(
            MCPClient(
                command="uv", args=["run", "mcp_server.py"], roots=root_paths
            )
        )
        clients["doc_client"] = doc_client

        chat = CliChat(
            doc_client=doc_client,
            clients=clients,
            claude_service=claude_service,
        )

        cli = CliApp(chat)
        await cli.initialize()
        await cli.run()
```

### Creating root objects

According to the **MCP spec**, all roots should have a URI that begins with `file://`.

This function takes the list of paths of that the user provided and turns them into `Root` objects.

```python
# mcp_client.py
class MCPClient:
    def __init__(
        self,
        command: str,
        args: list[str],
        env: Optional[dict] = None,
        roots: Optional[list[str]] = None,
    ):
        self._command = command
        self._args = args
        self._env = env
        self._roots = self._create_roots(roots) if roots else []
        self._session: Optional[ClientSession] = None
        self._exit_stack: AsyncExitStack = AsyncExitStack()
    '''============================================================='''
    def _create_roots(self, root_paths: list[str]) -> list[Root]:
        """Convert path strings to Root objects."""
        roots = []
        for path in root_paths:
            p = Path(path).resolve()
            file_url = FileUrl(f"file://{p}")
            roots.append(Root(uri=file_url, name=p.name or "Root"))
        return roots
    '''============================================================='''
```

### Roots callback

The client doesn't immediately provide the list of roots to the server. Instead, the server can make a request to the client at some future point in time. We make a callback that will be executed when the server requests the roots. The callback needs to return the list of roots inside of a `ListRootsResult` object.

This callback is passed into the `ClientSession` down on line `58`.

```python
# mcp_client.py
'''====================================================='''
async def _handle_list_roots(
    self, context: RequestContext["ClientSession", None]
) -> ListRootsResult | ErrorData:
    """Callback for when server requests roots."""
    return ListRootsResult(roots=self._roots)
'''====================================================='''
```

### Using the roots

On to the server. The server will use the roots in two scenarios:

1. Whenever a tool attempts to access a file or folder
2. When a LLM (like Claude) needs to resolve a file or folder to a full path. Think of when a user says 'read the todos.txt file' - Claude needs to figure out where the text file is, and might do so by looking at the list of roots

To handle the second case, we can either **define a tool that lists out the roots** or **inject them directly in a prompt**.

```python
# mcp_server.py
'''============================================================================'''
@mcp.tool()
async def list_roots(ctx: Context):
    """
    List all directories that are accessible to this server.
    These are the root directories where files can be read from or written to.
    """
    roots_result = await ctx.session.list_roots()
    client_roots = roots_result.roots

    return [file_url_to_path(root.uri) for root in client_roots]
'''============================================================================'''
```

### Accessing the roots

Roots are accessed by calling `ctx.session.list_roots()`.

This sends a message back to the client, which causes it to run the root-listing callback.

```python
@mcp.tool()
async def list_roots(ctx: Context):
    """
    List all directories that are accessible to this server.
    These are the root directories where files can be read from or written to.
    """
    '''=============================================='''
    roots_result = await ctx.session.list_roots()
    client_roots = roots_result.roots
    '''=============================================='''

    return [file_url_to_path(root.uri) for root in client_roots]
```

### Authorizing access

> **Remember: the MCP SDK does not attempt to limit what files or folders your tools attempt to read! You must implement that check yourself.**

Consider implementing a function like `is_path_allowed`, which will decide whether a path is accessible by comparing it to the list of roots.

```python
# mcp_server.py
'''===================================================================='''
async def is_path_allowed(requested_path: Path, ctx: Context) -> bool:
    roots_result = await ctx.session.list_roots()
    client_roots = roots_result.roots

    if not requested_path.exists():
        return False

    if requested_path.is_file():
        requested_path = requested_path.parent

    for root in client_roots:
        root_path = file_url_to_path(root.uri)
        try:
            requested_path.relative_to(root_path)
            return True
        except ValueError:
            continue

    return False
'''===================================================================='''
```

### Authorizing access

Once you've put an authorization function together - like `is_path_allowed` - use it throughout your tools to ensure the requested path is accessible.

```python
# mcp_server.py
@mcp.tool()
async def convert_video(
    input_path: str = Field(description="Path to the input MP4 file"),
    format: str = Field(description="Output format (e.g. 'mov')"),
    *,
    ctx: Context,
):
    """Convert an MP4 video file to another format using ffmpeg"""
    input_file = VideoConverter.validate_input(input_path)

    '''=================================================================='''
    # Ensure the input file is contained in a root
    if not await is_path_allowed(input_file, ctx):
        raise ValueError(f"Access to path is not allowed: {input_path}")
    '''=================================================================='''

    return await VideoConverter.convert(input_path, format)
```